In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    LongType,
    StructField,
    StructType,
    TimestampType,
)

CATALOG = "rio_dataengineering"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
AUDIT_SCHEMA = "audit"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {AUDIT_SCHEMA}")

PIPELINE_NAME = "people-analytics-lakehouse-pipeline"
RUN_TIMESTAMP = datetime.now()

print("Data-quality configuration ready.")

In [0]:
quality_results = []


def record_check(
    layer: str,
    table_name: str,
    check_name: str,
    failure_count: int,
) -> None:
    """Add one data-quality result to the audit collection."""

    quality_results.append(
        {
            "pipeline_name": PIPELINE_NAME,
            "layer": layer,
            "table_name": table_name,
            "check_name": check_name,
            "failure_count": int(failure_count),
            "status": "PASS" if failure_count == 0 else "FAIL",
            "checked_at": RUN_TIMESTAMP,
        }
    )

In [0]:
EXPECTED_BRONZE_ROWS = {
    "employees": 100,
    "departments": 7,
    "locations": 8,
    "payroll": 1052,
    "training": 300,
    "engagement": 70,
    "leave": 200,
}

for table_name, expected_rows in EXPECTED_BRONZE_ROWS.items():
    table = spark.table(
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )

    actual_rows = table.count()

    record_check(
        layer="bronze",
        table_name=table_name,
        check_name=f"expected_row_count_{expected_rows}",
        failure_count=abs(actual_rows - expected_rows),
    )

print("Bronze completeness checks prepared.")

In [0]:
silver_employees = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.employees"
)

record_check(
    "silver",
    "employees",
    "employee_id_not_null",
    silver_employees
    .filter(F.col("employee_id").isNull())
    .count(),
)

record_check(
    "silver",
    "employees",
    "employee_id_unique",
    silver_employees
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count(),
)

record_check(
    "silver",
    "employees",
    "birth_date_not_future",
    silver_employees
    .filter(F.col("birth_date") > F.current_date())
    .count(),
)

record_check(
    "silver",
    "employees",
    "hire_date_not_future",
    silver_employees
    .filter(F.col("hire_date") > F.current_date())
    .count(),
)

record_check(
    "silver",
    "employees",
    "termination_after_hire",
    silver_employees
    .filter(
        F.col("termination_date").isNotNull()
        & (F.col("termination_date") < F.col("hire_date"))
    )
    .count(),
)

record_check(
    "silver",
    "employees",
    "age_between_18_and_75",
    silver_employees
    .filter(~F.col("age_years").between(18, 75))
    .count(),
)

record_check(
    "silver",
    "employees",
    "status_logic_valid",
    silver_employees
    .filter(F.col("status_quality_flag") != "VALID")
    .count(),
)

print("Silver employee checks prepared.")

In [0]:
employee_profile = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.employee_profile"
)

record_check(
    "silver",
    "employee_profile",
    "department_join_complete",
    employee_profile
    .filter(F.col("department_name").isNull())
    .count(),
)

record_check(
    "silver",
    "employee_profile",
    "location_join_complete",
    employee_profile
    .filter(F.col("city").isNull())
    .count(),
)

record_check(
    "silver",
    "employee_profile",
    "employee_profile_unique",
    employee_profile
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count(),
)

print("Silver relationship checks prepared.")

In [0]:
silver_payroll = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.payroll"
)

record_check(
    "silver",
    "payroll",
    "employee_id_not_null",
    silver_payroll
    .filter(F.col("employee_id").isNull())
    .count(),
)

record_check(
    "silver",
    "payroll",
    "unique_employee_month",
    silver_payroll
    .groupBy("employee_id", "payroll_month")
    .count()
    .filter(F.col("count") > 1)
    .count(),
)

record_check(
    "silver",
    "payroll",
    "base_salary_non_negative",
    silver_payroll
    .filter(F.col("base_salary") < 0)
    .count(),
)

record_check(
    "silver",
    "payroll",
    "gross_pay_non_negative",
    silver_payroll
    .filter(F.col("gross_pay") < 0)
    .count(),
)

record_check(
    "silver",
    "payroll",
    "net_pay_not_above_gross_pay",
    silver_payroll
    .filter(F.col("net_pay") > F.col("gross_pay"))
    .count(),
)

print("Payroll checks prepared.")

In [0]:
silver_training = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.training"
)

silver_engagement = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.engagement"
)

silver_leave = spark.table(
    f"{CATALOG}.{SILVER_SCHEMA}.leave"
)

record_check(
    "silver",
    "training",
    "training_quality_valid",
    silver_training
    .filter(F.col("training_quality_flag") != "VALID")
    .count(),
)

record_check(
    "silver",
    "training",
    "score_between_0_and_100",
    silver_training
    .filter(~F.col("score").between(0, 100))
    .count(),
)

record_check(
    "silver",
    "engagement",
    "engagement_quality_valid",
    silver_engagement
    .filter(F.col("engagement_quality_flag") != "VALID")
    .count(),
)

record_check(
    "silver",
    "engagement",
    "overall_score_between_1_and_10",
    silver_engagement
    .filter(~F.col("overall_score").between(1, 10))
    .count(),
)

record_check(
    "silver",
    "leave",
    "leave_quality_valid",
    silver_leave
    .filter(F.col("leave_quality_flag") != "VALID")
    .count(),
)

record_check(
    "silver",
    "leave",
    "end_date_not_before_start_date",
    silver_leave
    .filter(F.col("end_date") < F.col("start_date"))
    .count(),
)

print("Subject-area checks prepared.")

In [0]:
workforce = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.workforce_overview"
)

employee_analytics = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.employee_analytics"
)

department_kpis = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.department_kpis"
)

location_kpis = spark.table(
    f"{CATALOG}.{GOLD_SCHEMA}.location_kpis"
)

record_check(
    "gold",
    "workforce_overview",
    "single_snapshot_row",
    abs(workforce.count() - 1),
)

record_check(
    "gold",
    "workforce_overview",
    "headcount_reconciles",
    workforce
    .filter(
        F.col("total_employees")
        != (
            F.col("active_employees")
            + F.col("terminated_employees")
        )
    )
    .count(),
)

record_check(
    "gold",
    "employee_analytics",
    "one_row_per_employee",
    employee_analytics
    .groupBy("employee_id")
    .count()
    .filter(F.col("count") > 1)
    .count(),
)

record_check(
    "gold",
    "department_kpis",
    "termination_rate_between_0_and_100",
    department_kpis
    .filter(~F.col("termination_rate_pct").between(0, 100))
    .count(),
)

record_check(
    "gold",
    "location_kpis",
    "location_id_not_null",
    location_kpis
    .filter(F.col("location_id").isNull())
    .count(),
)

print("Gold business checks prepared.")

In [0]:
audit_schema = StructType(
    [
        StructField("pipeline_name", StringType(), False),
        StructField("layer", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("check_name", StringType(), False),
        StructField("failure_count", LongType(), False),
        StructField("status", StringType(), False),
        StructField("checked_at", TimestampType(), False),
    ]
)

quality_df = spark.createDataFrame(
    quality_results,
    schema=audit_schema,
)

display(
    quality_df.orderBy(
        "status",
        "layer",
        "table_name",
        "check_name",
    )
)

In [0]:
(
    quality_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        f"{CATALOG}.{AUDIT_SCHEMA}.data_quality_results"
    )
)

print(
    f"Saved {quality_df.count()} quality-check results "
    f"to audit.data_quality_results."
)

In [0]:
failed_checks = (
    quality_df
    .filter(F.col("status") == "FAIL")
    .collect()
)

if failed_checks:
    failed_names = [
        (
            f"{row['layer']}."
            f"{row['table_name']}."
            f"{row['check_name']}"
        )
        for row in failed_checks
    ]

    raise ValueError(
        "Data-quality validation failed: "
        + ", ".join(failed_names)
    )

print("All Bronze, Silver and Gold quality checks passed.")